# 1 · Meet the Beast

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=01-meet-the-beast.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/01-meet-the-beast.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time). Use the ⚙ **View options** on
the site to switch story / how-to / quizzes / gimmicks on or off.</sub>
:::

:::{dropdown} 🎭 Story — your companion creature
:class: storytelling

*You are an adventurer, and every adventurer keeps a companion creature. You were
entrusted with **the Beast** — powerful, but not yet yours to command. Today the
masters simply **introduce** you: they show, on the Beast itself, what it can do.
They warm it from within, then grab a metal plate and stretch it to show how
robustly the Beast handles even large deformations. You only watch — the real
training starts next unit.*
:::

Almost every NGSolve session is the **same three steps**, no matter how hard the
problem: **(1)** conjure a **geometry and mesh**, **(2)** **solve a variational
problem** on it, **(3)** **visualize** the result. We will walk that loop **twice** —
once for a linear **Poisson** problem and once for a **nonlinear elasticity** problem —
so you see the shape of everything to come. Features fly past (boundary conditions,
solvers, nonlinear iterations); we name them and move on. Each gets its own unit later.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
# --- Bring the website's UI into this live notebook: the ⚙ View-options panel,
# the foldable story/how-to/quiz/further-reading categories and the gimmicks
# (rolling logo + winking head). Loads static/custom.css + view-options.js via
# notebooks/data/ngsum_ui.py. A no-op on the static-site build. --------------
import os
if not os.environ.get("WEBGUI_SCENE_DIR"):
    sys.path.insert(0, os.path.join(os.getcwd(), "data"))
    try:
        import ngsum_ui; ngsum_ui.enable()
    except Exception:
        pass

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve import solvers, preconditioners
from ngsolve.krylovspace import LinearSolverCreator, CGSolver
from ngsolve.webgui import Draw

## 1. Geometry & mesh — the Beast

The Beast is NGSolve's logo **sculpture**, carved from just two primitive shapes — a
**sphere** and a **cylinder** — with **boolean subtraction**. Read the four panels of the
sketch below, left to right:

1. **A solid sphere** — start from a full ball.
2. **Hollow it into a shell** — subtract a smaller, concentric sphere, leaving a thick
   spherical shell.
3. **Line up three bores** — three mutually orthogonal cylinders, aimed straight through the
   shell along the $x$-, $y$- and $z$-axes.
4. **The Beast** — subtract those three cylinders; what remains is the sculpture.

```{image} data/beast-construction.jpg
:alt: Building the Beast in four steps — a solid sphere, hollowed into a shell, three bores lined up, the finished sculpture
:width: 760px
:align: center
```

We build exactly that with **Netgen/OCC** constructive geometry below — the four code
comments are the four panels. Naming a few faces now lets us attach boundary conditions later.

In [ ]:
def beast_sculpture():
    centre = Pnt(50, 50, 50)
    ball  = Sphere(centre, 80)                                   # 1. a solid sphere
    shell = ball - Sphere(centre, 50)                            # 2. hollow it → a thick shell
    bores = [Cylinder(Pnt(-100,    0,    0), X, r=40, h=300),    # 3. three orthogonal bores,
             Cylinder(Pnt( 100, -100,  100), Y, r=40, h=300),    #    aimed along x, y and z
             Cylinder(Pnt(   0,  100, -100), Z, r=40, h=300)]
    beast = shell
    for bore in bores:                                           # 4. drill them out → the Beast
        beast = beast - bore
    return beast.Move((-50, -50, -50)).Scale(Pnt(0, 0, 0), 0.05)  # centre + shrink

beast = beast_sculpture()
mesh = Mesh(OCCGeometry(beast).GenerateMesh(maxh=0.7))
mesh.Curve(2)
print(f"the Beast: {mesh.nv} vertices, {mesh.ne} elements")
Draw(mesh)

## 2. A variational solve — fire-energy inside the Beast

Legend says the Beast stores the energy for its fire-breath deep in its body. We model
that as a **heat source** living inside the shell and ask for the steady temperature —
the **Poisson problem** $-\Delta u = f$ with $u=0$ on the surface. In NGSolve this is a
**weak form** on an `H1` space: find $u$ such that
$$ \underbrace{\int_\Omega \nabla u\cdot\nabla v}_{a(u,\,v)} \;=\; \underbrace{\int_\Omega f\,v}_{f(v)} \qquad\text{for all } v. $$
Recent NGSolve lets you write *exactly that* — the weak form as `a(u, v) == f(v)` — and
hand it to a single **`Solve`** that returns the solution. (Inhomogeneous boundary data can
be given **inline**, e.g. `u[BND("name")] == value`; here it is simply zero — more in unit 5.)

```{image} data/beast-heatsource.png
:alt: A slice through the Beast's shell showing the heat source f as a bright band just beneath the outer surface
:width: 380px
:align: center
```
<p style="text-align:center"><sub>A slice through the shell: the heat source <code>f</code>
sits in a thin band <em>just beneath</em> the Beast's surface (where <code>u = 0</code>), in
the solid material between the three bores. The solve spreads it inward; afterwards we clip
the shell open to look right at the hot spot.</sub></p>

In [ ]:
r = sqrt(x*x + y*y + z*z)                            # the shell spans radii 2.5 … 4.0
source = 40 * exp(-((r - 3.7) / 0.22)**2)            # a glow just beneath the outer surface

u, v = H1(mesh, order=2, dirichlet=".*").TnT()       # trial & test fns; u = 0 on the surface

# The compact variational solve: write the weak form as `a(u,v) == f(v)` and hand it to
# `Solve`, which returns the solution GridFunction. The two trailing arguments pin a
# **CI-safe iterative** solver — a local preconditioner with CG — so the solve never touches
# the (here flaky) direct factorisations; drop them to fall back to a direct solve. Solvers
# get their own unit (6).
gfu = Solve(grad(u) * grad(v) * dx == source * v * dx,
            preconditioners.Local(), LinearSolverCreator(CGSolver, maxiter=2000))
print(f"hottest point inside the Beast: {max(gfu.vec):.2f}")

# Clip the shell open (a plane through the centre, normal −z) so we can look right inside
# at the hot spot instead of only seeing the cold u = 0 surface.
Draw(gfu, mesh, "temperature",
     clipping={"function": True, "pnt": (0, 0, 0), "vec": (0, 0, -1)})

### The same solve, unpacked — meet the detail objects

That one-liner is convenient, but the rest of this tutorial works directly with the
**objects it hides**, so let us build them **by hand** and name them. The compact `Solve`
essentially makes a **`BilinearForm`** (→ a sparse **matrix** `a.mat`) and a
**`LinearForm`** (→ a **load vector** `f.vec`), then solves the linear system on the
**free** (non-Dirichlet) dofs — `fes.FreeDofs()`. Spelled out, with a direct factorisation:

In [ ]:
fes = H1(mesh, order=2, dirichlet=".*")              # the space the compact form built for us
u, v = fes.TnT()                                     # trial & test functions
a = BilinearForm(grad(u) * grad(v) * dx).Assemble()  # BilinearForm  →  sparse matrix  a.mat
f = LinearForm(source * v * dx).Assemble()           # LinearForm    →  load vector    f.vec
gfu_manual = GridFunction(fes)                        # its coefficients live in .vec
gfu_manual.vec.data = \
    a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec   # solve on the free dofs
print(f"by hand, the same hottest point: {max(gfu_manual.vec):.2f}")

Same answer, two routes. These pieces — **`BilinearForm`**, **`LinearForm`**,
**`GridFunction`**, the free dofs, and the assembled **matrix**/**vector** with its
**`Inverse`** — are the vocabulary of every later unit. `Solve(a(u,v) == f(v), …)` is just
the shorthand once they are familiar; the *iterative* route it took above (preconditioner
+ Krylov) gets unpacked in unit 6.

## 3. The same loop, nonlinear — stretching a Swiss-cross plate

To show off **robustness**, the masters grab a stiff specimen — a **Swiss-cross plate**, a
landscape slab with a cross punched clean through — clamp its left edge and **pull** the
right. This third pass is a genuine *teaser*: it shows the same loop carrying a much harder
problem, with the details deferred to later units. First, before any code, the **model**.

**Geometry.** A thin plate $\Omega\subset\mathbb{R}^3$ (width × height × thickness) with a
cross-shaped hole. Its left face $\Gamma_{\mathrm{hold}}$ is **clamped**; its right face
$\Gamma_{\mathrm{pull}}$ carries a horizontal **pulling traction** of strength $g$.

```{image} data/elasticity-plate.png
:alt: The Swiss-cross plate — a red slab with a cross-shaped hole, clamped at the left, pulled to the right
:width: 380px
:align: center
```
<p style="text-align:center"><sub>The specimen: a red Swiss-cross plate (the cross is a hole
through to the white background), clamped at the left edge and pulled to the right.</sub></p>

**Kinematics.** The unknown is a **displacement** $\mathbf{u}:\Omega\to\mathbb{R}^3$ moving
each material point $\mathbf{x}\mapsto\mathbf{x}+\mathbf{u}$. Its **deformation gradient** is
$F = I + \nabla\mathbf{u}$, with $J=\det F>0$ and right Cauchy–Green tensor $C=F^{\top}F$.

**Material.** A compressible **Neo-Hookean** hyperelastic stored-energy density
$$ \psi(F) \;=\; \tfrac{\mu}{2}\bigl(\operatorname{tr}C - 3\bigr)\;-\;\mu\,\log J\;+\;\tfrac{\lambda}{2}(\log J)^2,
   \qquad \mu=\frac{E}{2(1+\nu)},\quad \lambda=\frac{E\,\nu}{(1+\nu)(1-2\nu)} . $$

**Variational problem.** The plate settles into the displacement that **minimises the total
potential energy** — stored energy minus the work of the traction — over all admissible
(clamped) displacements:
$$ \mathbf{u}=\arg\min_{\substack{\mathbf{v}\in[H^1(\Omega)]^3\\[1pt]\mathbf{v}=0\ \text{on }\Gamma_{\mathrm{hold}}}}
   \;\Bigl[\;\int_\Omega \psi\bigl(I+\nabla\mathbf{v}\bigr)\,\mathrm{d}x\;-\;\int_{\Gamma_{\mathrm{pull}}} g\,v_1\,\mathrm{d}s\;\Bigr]. $$
Setting its first variation to zero gives a **nonlinear** system; we solve it by **Newton's
method** (`solvers.Newton`), **ramping** $g$ up from $0$ in small steps so each Newton solve
starts near its basin. The same *geometry → weak form → solve → draw* loop — only the weak
form is an **energy** now, and the solve **iterates**.

In [ ]:
Wx, Wy, t, bw = 9.0, 6.0, 0.5, 1.0                   # a landscape plate: width × height × thickness
slab = Box(Pnt(0, 0, 0), Pnt(Wx, Wy, t))
cx, cy = Wx / 2, Wy / 2
vbar = Box(Pnt(cx - bw/2, cy - 2.0, -0.1), Pnt(cx + bw/2, cy + 2.0, t + 0.1))   # vertical cross bar
hbar = Box(Pnt(cx - 3.0, cy - bw/2, -0.1), Pnt(cx + 3.0, cy + bw/2, t + 0.1))   # horizontal cross bar
plate = slab - vbar - hbar                           # the Swiss-cross plate: a cross punched out
plate.faces.Min(X).name = "hold"                     # clamped left edge   Γ_hold
plate.faces.Max(X).name = "pull"                     # pulled right edge   Γ_pull
fmesh = Mesh(OCCGeometry(plate).GenerateMesh(maxh=0.7)); fmesh.Curve(1)

E, nu = 200.0, 0.35                                  # Young's modulus, Poisson ratio
mu, lam = E / (2 * (1 + nu)), E * nu / ((1 + nu) * (1 - 2 * nu))
V = VectorH1(fmesh, order=1, dirichlet="hold")
ud = V.TrialFunction()
F = Id(3) + Grad(ud); J = Det(F); C = F.trans * F
psi = 0.5 * mu * (Trace(C) - 3) - mu * log(J) + 0.5 * lam * log(J)**2   # Neo-Hooke

traction = Parameter(0.0)
elastic = BilinearForm(V, symmetric=True)
elastic += Variation(psi * dx)                       # stored elastic energy
elastic += Variation(-traction * ud[0] * ds("pull"))  # work of the pulling traction

gfd = GridFunction(V); gfd.vec[:] = 0
morph = GridFunction(V); morph.vec[:] = 0            # frame 0: the original, undeformed flag
nsteps = 12                                          # many small load steps → a slow, smooth morph
for k in range(1, nsteps + 1):                       # ramp the load, keeping continuation
    traction.Set(22.0 * k / nsteps)
    solvers.Newton(elastic, gfd, inverse="sparsecholesky", dampfactor=0.5, printing=False)
    morph.AddMultiDimComponent(gfd.vec)              # one morph frame per load step
print(f"the plate stretched by {max(abs(gfd.vec.FV().NumPy())):.2f} units — and held")

We draw it as a **morph**: a multidim field whose frames are the plate at **successive load
steps**, from undeformed to fully pulled. In the webgui, press **play** (or drag the
**multidim** slider) to watch it stretch — the many frames make the animation slow and
smooth, the surface warps with the displacement and is coloured by its magnitude.

In [ ]:
Draw(morph, fmesh, "displacement", deformation=True,
     interpolate_multidim=True, animate=True)

:::{dropdown} 📚 Further reading
:class: further-reading

- **This unit is only a teaser — the rest of the course is the "further reading".** Each
  thing that flew past gets its own unit here: **Dirichlet data & the linear solve** in
  [unit 5](05-solving.ipynb), the **solver toolbox** behind `Solve`/`Newton` in
  [unit 6](06-linear-solvers.ipynb), linear **elasticity** in [unit 13](13-elasticity.ipynb),
  and a **coupled, nonlinear** thermo-elastic bar in [unit 16](16-thermoelasticity.ipynb).
- **The Poisson solve, step by step** — i-tutorial
  [1.1 First NGSolve example](https://docu.ngsolve.org/latest/i-tutorials/unit-1.1-poisson/poisson.html).
- **Hyperelasticity & Newton** — i-tutorial
  [3D Solid Mechanics](https://docu.ngsolve.org/latest/i-tutorials/wta/elasticity3D.html)
  and the Newton loop in
  [3.3 Nonlinear problems](https://docu.ngsolve.org/latest/i-tutorials/unit-3.3-nonlinear/nonlinear.html);
  a fuller derivation in the
  [SciCADE course — 3D elasticity](https://jschoeberl.github.io/SciCADE-course/unit2-elasticity/elasticity3D.html).
- **The Poisson problem, rigorously** — the weak form, Sobolev spaces and Lax–Milgram in
  Schöberl's iFEM [weak formulation of the Poisson equation](https://jschoeberl.github.io/iFEM/sobolevspaces/preciseweak.html).
:::

:::{dropdown} 🧠 Quiz — `Redraw()` vs `scene.Redraw()`?
:class: quiz

To animate a time- or load-loop you refresh the picture each step — but with *which* call?

- **`scene = Draw(...)`** in a **Jupyter notebook** hands back a **webgui scene object**.
  Calling **`scene.Redraw()`** pushes the new data into *that* embedded 3-D view in the
  output cell. This is the in-notebook way — JupyterLite, Colab and local Jupyter all use it.
- The global **`Redraw()`** (i.e. `ngsolve.Redraw()`) instead refreshes the **Netgen GUI**
  desktop window — the viewer you get when a script is launched through `netgen myscript.py`,
  or as a plain `python myscript.py` after `import netgen.gui`.

Rule of thumb: **in-notebook webgui → `scene.Redraw()`** on the object `Draw` returned;
**Netgen desktop GUI → global `Redraw()`**. (Above we instead let the webgui animate a
pre-computed `multidim` field via `animate=True`, so no manual redraw is needed at all.)
:::

**Next:** the masters withdraw. To approach the Beast yourself you first conjure it some
**food** — and learn elementary **geometry & meshing** along the way (unit 2).

In [ ]:
# Navigation to the next unit — shown only in a live notebook (Colab /
# JupyterLite / local Jupyter), never in the rendered website.
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _nb, _title = "02-geometry", "2 · Conjuring geometry & taming the mesh 🍫"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:                                           # JupyterLite & local open relative .ipynb links
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))